In [ ]:
import zipfile
import xml.etree.ElementTree as ET
import os

namespaces = {
    'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'
}

def extract_content(docx_path):
    out = []
    with zipfile.ZipFile(docx_path) as docx:
        xml_content = docx.read('word/document.xml')
        root = ET.fromstring(xml_content)
        
        body = root.find('{' + namespaces['w'] + '}body')
        if body is None:
            return "No body found"
            
        for child in body:
            # Paragraph
            if child.tag == '{' + namespaces['w'] + '}p':
                p_text = []
                for run in child.iter('{' + namespaces['w'] + '}t'):
                    if run.text:
                        p_text.append(run.text)
                out.append("".join(p_text))
            # Table
            elif child.tag == '{' + namespaces['w'] + '}tbl':
                out.append("\n")
                first_row = True
                for row in child.iter('{' + namespaces['w'] + '}tr'):
                    row_cells = []
                    for cell in row.iter('{' + namespaces['w'] + '}tc'):
                        cell_text = []
                        for run in cell.iter('{' + namespaces['w'] + '}t'):
                            if run.text:
                                cell_text.append(run.text)
                        row_cells.append("".join(cell_text).replace("\n", " ").strip())
                    out.append("| " + " | ".join(row_cells) + " |")
                    if first_row:
                        out.append("|" + "---|"*len(row_cells))
                        first_row = False
                out.append("\n")
    return "\n".join(out)

docx_path = r"d:\flowsync\FlowSync_需求规格说明书.docx"
output_path = r"d:\flowsync\requirements.md"

try:
    content = extract_content(docx_path)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Successfully wrote requirements to requirements.md!")
except Exception as e:
    print(f"Error: {e}")